# **NEURAL CTR**

Neural Click-Through-Rate. In this task, given a data set of 39 features containing info about an ad's impression and response amomng users, we know have to predict binary labels indicating 1 (user clicked) or 0 (user skipped).

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss, average_precision_score, f1_score

# Data Pre-processing

In [ ]:
def preprocess_ctr_data(csv_path):
    print("Loading data...")

    df = pd.read_csv(csv_path)
    
    numeric_cols = [f'I{i}' for i in range(1, 14)]
    categorical_cols = [f'C{i}' for i in range(1, 27)]

    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
    
    print("Processing Numeric Features...")
    for col in numeric_cols:
        median_val = train_df[col].median()
        train_df[col] = train_df[col].fillna(median_val)
        val_df[col] = val_df[col].fillna(median_val)
   
        min_val = train_df[col].min()
        shift = abs(min_val) + 1 if min_val < 0 else 1
        train_df[col] = np.log1p(train_df[col] - min_val if min_val < 0 else train_df[col])
        val_df[col] = np.log1p(val_df[col] - min_val if min_val < 0 else val_df[col])

    scaler = StandardScaler()
    train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
    val_df[numeric_cols] = scaler.transform(val_df[numeric_cols]) # Only transform val!

    print("Processing Categorical Features...")
    train_df[categorical_cols] = train_df[categorical_cols].fillna('<UNK>')
    val_df[categorical_cols] = val_df[categorical_cols].fillna('<UNK>')

    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    train_df[categorical_cols] = encoder.fit_transform(train_df[categorical_cols])
    val_df[categorical_cols] = encoder.transform(val_df[categorical_cols])

    train_df[categorical_cols] = train_df[categorical_cols] + 1
    val_df[categorical_cols] = val_df[categorical_cols] + 1

    cat_dims = [int(train_df[col].max() + 2) for col in categorical_cols]
    
    return train_df, val_df, numeric_cols, categorical_cols, cat_dims